# Georgiev et al. comparison + three-way validation

Overplots this pipeline's three kSZ results (stitched, coeval-direct,
coeval-Georgiev-reconstruction) against **Georgiev et al.'s own published
parameter-study data** (`obs_vs_params_xe_ksz_data.pickle`) -- a semi-analytic
model swept over 4 reionization-history parameters (`zre`, `zend`, `alpha0`,
`kappa`), 3 test values each, giving 12 (xe(z), D_ell) curve pairs total.

**Naming note, read before editing:** "coeval-Georgiev" elsewhere in this
repo means *our own* reconstruction of P_qperp from Pee/Pvv/Pev via the
Georgiev+24 Eq.(10) convolution (see `georgiev_convolution.py`) -- that is
NOT the same thing as the data loaded in this notebook, which is Georgiev
et al.'s actual published model curves. Both appear on the same plots below;
they are labeled distinctly (`ours (Georgiev-Eq10 recon)` vs
`Georgiev+24 (param sweep)`) specifically to avoid conflating them.

No new science logic here (per `README.md`'s "no science logic" convention
for `notebooks/exploratory/`) -- this only loads already-computed products
and plots/compares them.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors

plt.rcParams.update({'font.family': 'serif', 'font.size': 12, 'axes.linewidth': 1.3})

# ---- paths -- ADJUST THESE if your layout differs ----
PRODUCTS_DIR = "../../data/products"
GEORGIEV_PICKLE = "../../data/external/georgiev/obs_vs_params_xe_ksz_data.pickle"
# ^ TODO: this repo has no data/external/ convention yet -- place the pickle
#   here (or wherever you prefer) and update this path. Keep it out of
#   data/products/, since that's specifically this pipeline's OWN outputs.

In [ ]:
# ---- load our own three results ----
coeval   = np.load(f"{PRODUCTS_DIR}/ksz_Dl_coeval.npz")
georgiev_recon = np.load(f"{PRODUCTS_DIR}/ksz_Dl_coeval_georgiev.npz")  # OUR reconstruction, not Georgiev's data
stitched = np.load(f"{PRODUCTS_DIR}/ksz_Dl_stitched.npz")
reion    = np.load(f"{PRODUCTS_DIR}/coeval_reion.npz")

print("coeval-direct   D_3000 =", float(np.interp(3000, coeval['ell'], coeval['Dl'])), "uK^2")
print("coeval-Georgiev D_3000 =", float(np.interp(3000, georgiev_recon['ell'], georgiev_recon['Dl'])), "uK^2  (our Eq.10 reconstruction)")
print("stitched        D_3000 =", float(np.interp(3000, stitched['ell'], stitched['Dl'])), "uK^2")
print("coeval_reion z-range (patchy):", reion['z'].min(), "-", reion['z'].max())

In [ ]:
# ---- load Georgiev et al.'s published parameter-study data ----
with open(GEORGIEV_PICKLE, 'rb') as f:
    geo = pickle.load(f)

zlin        = geo['zlin']          # (100,) redshift grid, 0-20
lrange      = geo['lrange']        # (100,) multipole grid, 100-10000 (not evenly log-spaced)
ndeg        = geo['ndeg']          # 4 parameters
ntest       = geo['ntest']         # 3 test values per parameter
labels      = geo['labels']        # LaTeX labels, e.g. r'$z_\mathrm{re}$'
paramnames  = geo['paramnames']    # ['zre', 'zend', 'alpha0', 'kappa']
results     = geo['results']       # dict keyed 0..ndeg-1

print(f"{ndeg} parameters x {ntest} test values each:", paramnames)

## D_ell(ell) overlay

Each row is one of Georgiev's 4 swept parameters (others held fixed);
each faint line is one of their 3 test values for that parameter. Our
three curves (same on every row, since they don't vary by *their*
parameter grid) show where this pipeline's fiducial run sits relative
to the region of parameter space they explored.

In [ ]:
fig, axes = plt.subplots(ndeg, 1, figsize=(7, 3.2 * ndeg), sharex=True)

for i in range(ndeg):
    ax = axes[i]
    cmap = cm.get_cmap('PuRd')
    norm = colors.Normalize(vmin=geo['params_test'][:, i].min(),
                             vmax=geo['params_test'][:, i].max())

    for u in range(ntest):
        theta = results[i]['theta'][u]
        ax.plot(lrange, results[i]['ksz'][u], color=cmap(0.35 + 0.55 * norm(theta[i])),
                 lw=1.5, label=f"{paramnames[i]}={theta[i]:.3g}")

    # our three results, same on every row
    ax.plot(coeval['ell'], coeval['Dl'], color='k', lw=2.2, ls='-', label='ours: coeval-direct')
    ax.plot(stitched['ell'], stitched['Dl'], color='tab:blue', lw=2.0, ls='--', label='ours: stitched')
    ax.plot(georgiev_recon['ell'], georgiev_recon['Dl'], color='tab:orange', lw=2.0, ls=':',
             label='ours: Georgiev-Eq10 recon')

    # Reichardt+2021 anchor point
    ax.errorbar([3000], [1.1], yerr=[[0.7], [1.0]], fmt='*', color='green',
                 markersize=14, capsize=4, label='Reichardt+2021', zorder=10)

    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_ylabel(r"$\mathcal{D}_\ell^\mathrm{kSZ}$ [$\mu$K$^2$]")
    ax.set_title(f"varying {labels[i]}", fontsize=11)
    ax.legend(fontsize=8, ncol=2, loc='upper left')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel(r"Angular multipole $\ell$")
fig.tight_layout()
fig.savefig("../../data/plots/georgiev_comparison_Dell.pdf")
print("Saved -> data/plots/georgiev_comparison_Dell.pdf")

## Ionization history x_e(z) overlay

Same layout: Georgiev's swept parameter families vs. our own reionization
history from `coeval_reion.npz`.

In [ ]:
our_z  = reion['z']
our_xe = reion['xe']

fig, axes = plt.subplots(ndeg, 1, figsize=(7, 3.2 * ndeg), sharex=True)

for i in range(ndeg):
    ax = axes[i]
    cmap = cm.get_cmap('PuRd')
    norm = colors.Normalize(vmin=geo['params_test'][:, i].min(),
                             vmax=geo['params_test'][:, i].max())

    for u in range(ntest):
        theta = results[i]['theta'][u]
        ax.plot(zlin, results[i]['xe'][u], color=cmap(0.35 + 0.55 * norm(theta[i])),
                 lw=1.5, label=f"{paramnames[i]}={theta[i]:.3g}")

    ax.plot(our_z, our_xe, color='k', lw=2.2, label='ours (coeval, patchy range)')

    ax.set_ylabel(r"$x_e(z)$")
    ax.set_title(f"varying {labels[i]}", fontsize=11)
    ax.legend(fontsize=8, ncol=2, loc='lower left')
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 20)

axes[-1].set_xlabel(r"Redshift $z$")
fig.tight_layout()
fig.savefig("../../data/plots/georgiev_comparison_xe.pdf")
print("Saved -> data/plots/georgiev_comparison_xe.pdf")

## D_ell ratio diagnostics (for the validation table)

Same two open questions as the handoff doc:
1. stitched / coeval-direct -- flat vs ell (normalization) or ell-dependent (shape)?
2. coeval-direct / our-Georgiev-reconstruction -- confirming the ~3x overshoot, vs ell and vs redshift.

In [ ]:
def loglog_interp(xq, xp, fp):
    xp, fp = np.asarray(xp), np.asarray(fp)
    m = (xp > 0) & (fp > 0)
    lx, lf = np.log(xp[m]), np.log(fp[m])
    lq = np.log(np.clip(xq, xp[m].min(), xp[m].max()))
    return np.exp(np.interp(lq, lx, lf))

# 1. stitched vs coeval-direct
lo, hi = max(coeval['ell'].min(), stitched['ell'].min()), min(coeval['ell'].max(), stitched['ell'].max())
ell_c = coeval['ell'][(coeval['ell'] >= lo) & (coeval['ell'] <= hi)]
ratio_stitch = loglog_interp(ell_c, stitched['ell'], stitched['Dl']) / loglog_interp(ell_c, coeval['ell'], coeval['Dl'])

print("stitched/coeval-direct ratio: range", ratio_stitch.min(), "-", ratio_stitch.max(),
      " (max/min =", ratio_stitch.max()/ratio_stitch.min(), ")")
print(" -> FLAT (normalization)" if ratio_stitch.max()/ratio_stitch.min() < 1.3 else " -> ELL-DEPENDENT (shape)")

plt.figure(figsize=(6,4))
plt.plot(ell_c, ratio_stitch)
plt.axhline(1.0, color='gray', ls=':')
plt.xscale('log'); plt.xlabel(r"$\ell$"); plt.ylabel("stitched / coeval-direct")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("../../data/plots/ratio_stitched_vs_coeval.pdf")

In [ ]:
# 2. our Georgiev-reconstruction vs coeval-direct, vs ell and vs redshift
lo_g, hi_g = max(coeval['ell'].min(), georgiev_recon['ell'].min()), min(coeval['ell'].max(), georgiev_recon['ell'].max())
ell_g = coeval['ell'][(coeval['ell'] >= lo_g) & (coeval['ell'] <= hi_g)]
ratio_g_ell = loglog_interp(ell_g, coeval['ell'], coeval['Dl']) / loglog_interp(ell_g, georgiev_recon['ell'], georgiev_recon['Dl'])

print("direct/reconstructed ratio vs ell: range", ratio_g_ell.min(), "-", ratio_g_ell.max())

pee = np.load(f"{PRODUCTS_DIR}/pee_pvv_pev.npz")
print()
print("per-redshift direct/reconstructed ratio (pre-Limber, Pqperp level):")
for i, z in enumerate(pee['z']):
    key = f"ratio_{i}"
    if key in pee:
        r = pee[key]
        print(f"  z={z:5.1f}: mean={r.mean():.3f}  range=[{r.min():.3f}, {r.max():.3f}]")